<a href="https://colab.research.google.com/github/Ashok9021/data-sci/blob/main/Copy_of_ashok_data_sci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b1eba2efc52eb6bf0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# 1. Install necessary libraries (Run this in a separate cell if needed)
# !pip install transformers torch gradio spacy pandas scikit-learn
# !python -m spacy download en_core_web_sm

import torch
import gradio as gr
import pandas as pd
import spacy
import warnings
from transformers import pipeline
from sklearn.linear_model import LinearRegression

# Ignore warnings for a cleaner output
warnings.filterwarnings('ignore')

# 2. Initialize Models
# Zero-shot classification for intent recognition
classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

# Spacy for Named Entity Recognition (NER) to extract variables
try:
    nlp = spacy.load('en_core_web_sm')
except:
    # Fallback if model isn't downloaded
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

# 3. Helper Functions
def classify_prompt(prompt):
    labels = ["descriptive_statistics", "correlation", "regression", "others"]
    result = classifier(prompt, candidate_labels=labels)
    return result['labels'][0]

def extract_variables(prompt):
    doc = nlp(prompt)
    # Extracts GPE (Locations) or ORG (Organizations) as per your logic
    # Note: You might want to add 'PRODUCT' or 'CARDINAL' depending on your CSV headers
    variables = [ent.text for ent in doc.ents if ent.label_ == "GPE" or ent.label_ == "ORG"]
    if len(variables) == 0:
        return None
    return variables

def load_data(file):
    if not file.name.endswith('.csv'):
        raise ValueError("Please upload a CSV file.")
    return pd.read_csv(file.name)

def get_columns(df):
    return df.columns.tolist()

# 4. Analysis Functions
def descriptive_statistics(df, variables):
    return df[variables].describe()

def correlation(df, variables):
    return df[variables].corr()

def regression(df, dependent_var, independent_vars):
    X = df[independent_vars]
    y = df[dependent_var]
    model = LinearRegression()
    model.fit(X, y)
    return f"Coefficients: {model.coef_}, Intercept: {model.intercept_}"

# 5. Main Processing Logic
def analyze_data(file, prompt):
    analysis_type = classify_prompt(prompt)
    variables = extract_variables(prompt)

    if variables is None:
        return "Please specify the variables for analysis (e.g., mention column names)."

    try:
        df = load_data(file)
    except ValueError as e:
        return str(e)

    columns = get_columns(df)

    # Check if extracted variables actually exist in the CSV
    if set(variables).issubset(columns):
        if analysis_type == 'descriptive_statistics':
            result = descriptive_statistics(df, variables)
        elif analysis_type == 'correlation':
            result = correlation(df, variables)
        elif analysis_type == 'regression':
            if len(variables) < 2:
                return "Regression requires at least two variables (dependent and independent)."
            result = regression(df, variables[0], variables[1:])
        else:
            result = "Analysis type not recognized."
    else:
        return f"Some variables {variables} not found in the dataset columns: {columns}"

    return str(result)

# 6. Gradio Interface
interface = gr.Interface(
    fn=analyze_data,
    inputs=[
        gr.File(label="Upload CSV File"),
        gr.Textbox(label="Analysis Prompt", placeholder="e.g., Show me the correlation between Sales and Marketing")
    ],
    outputs="text",
    title="Gen AI Data Analysis Assistant"
)

if __name__ == "__main__":
    interface.launch(debug=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://16b9b94836334a2461.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
